# AI-Powered Gen Z Credit Card Product Innovation System
## Notebook 01 – Credit Card Product Intelligence Database

Description

This notebook constructs the Product Intelligence Database used throughout the product innovation pipeline. Synthetic Canadian credit card products are cleaned, transformed, and enriched using domain-specific feature engineering. Binary product features are aggregated into engineered feature-level scores representing travel, cashback, digital capabilities, subscriptions, lifestyle, sustainability, financial wellness, security, and premium benefits. The products are then clustered to identify major product identities within the Canadian credit card market.

Main Tasks
• Product data cleaning
• Removal of business credit cards
• Product feature engineering
• Calculation of engineered feature levels
• Product clustering
• Product identity interpretation

Output
• Product Intelligence Database
• Engineered product feature scores
• Product clusters
• Product identity labels

In [ ]:
# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv("synthetic_canadian_credit_card_product_intelligence_dataset.csv")

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

In [ ]:
# Product category distribution BEFORE removing Business cards

print(df["card_category"].value_counts())

In [ ]:
# ============================================================
# Remove Business Credit Cards
# ============================================================

print("Original dataset size:", len(df))

# Remove Business cards
df = df[df["card_category"] != "Business"].reset_index(drop=True)

print("Updated dataset size:", len(df))

print("\nRemaining Product Categories:")

print(df["card_category"].value_counts())

In [ ]:
# ============================================================
# Rename Engineered Product Intelligence Features
# ============================================================

df = df.rename(columns={
    "digital_focus_score": "digital_feature_level",
    "cashback_focus_score": "cashback_feature_level",
    "travel_focus_score": "travel_feature_level",
    "lifestyle_focus_score": "lifestyle_feature_level",
    "subscription_focus_score": "subscription_feature_level",
    "sustainability_focus_score": "sustainability_feature_level",
    "financial_wellness_score": "financial_wellness_feature_level"
})

print("Feature names updated successfully.")

print(df.columns)

In [ ]:
# ============================================================
# Remove previously generated engineered feature levels
# ============================================================

old_engineered_columns = [
    "travel_feature_level",
    "cashback_feature_level",
    "digital_feature_level",
    "subscription_feature_level",
    "lifestyle_feature_level",
    "sustainability_feature_level",
    "financial_wellness_feature_level"
]

df = df.drop(
    columns=[col for col in old_engineered_columns if col in df.columns],
    errors="ignore"
)

print("Old engineered feature levels removed.")

In [ ]:
df = df.drop(columns=["business_rewards"], errors="ignore")

In [ ]:
# ============================================================
# Add Security and Premium Benefit Features
# ============================================================

import numpy as np

rng = np.random.default_rng(42)

def generate_feature(probability, size):
    return rng.binomial(1, probability, size)


# -----------------------------
# Security and protection
# -----------------------------

# Common protections across most consumer cards
df["fraud_alerts"] = generate_feature(0.82, len(df))
df["zero_liability"] = generate_feature(0.90, len(df))

# More advanced protections are less common
df["identity_protection"] = generate_feature(0.36, len(df))
df["purchase_protection"] = generate_feature(0.48, len(df))


# -----------------------------
# Premium benefits
# -----------------------------

premium_mask = df["card_category"].eq("Premium Travel")
travel_mask = df["card_category"].isin(["Travel", "Premium Travel"])
high_fee_mask = df["annual_fee_cad"] >= 120

df["concierge_service"] = 0
df["premium_hotel_benefits"] = 0
df["luxury_travel_credits"] = 0
df["exclusive_event_access"] = 0

# Premium Travel products have the highest probability
df.loc[premium_mask, "concierge_service"] = rng.binomial(
    1, 0.75, premium_mask.sum()
)

df.loc[premium_mask, "premium_hotel_benefits"] = rng.binomial(
    1, 0.72, premium_mask.sum()
)

df.loc[premium_mask, "luxury_travel_credits"] = rng.binomial(
    1, 0.68, premium_mask.sum()
)

df.loc[premium_mask, "exclusive_event_access"] = rng.binomial(
    1, 0.55, premium_mask.sum()
)

# Some standard travel or high-fee cards may also include them
non_premium_high_value = (~premium_mask) & (travel_mask | high_fee_mask)

df.loc[non_premium_high_value, "concierge_service"] = rng.binomial(
    1, 0.15, non_premium_high_value.sum()
)

df.loc[non_premium_high_value, "premium_hotel_benefits"] = rng.binomial(
    1, 0.20, non_premium_high_value.sum()
)

df.loc[non_premium_high_value, "luxury_travel_credits"] = rng.binomial(
    1, 0.12, non_premium_high_value.sum()
)

df.loc[non_premium_high_value, "exclusive_event_access"] = rng.binomial(
    1, 0.10, non_premium_high_value.sum()
)

print("Security and premium raw features added.")

In [ ]:
df.to_csv(
    "synthetic_canadian_credit_card_product_intelligence_dataset_v2.csv",
    index=False
)

print("Updated dataset saved successfully.")